In [ ]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 42.5 MB/s eta 0:00:00


In [ ]:
%%writefile detector.py
import pandas as pd
import re

def parse_logs(file_obj):
    """Parses uploaded log file into a DataFrame and flags suspicious activity."""
    logs = []
    ip_pattern = re.compile(r'\b(?:[0-9]{1,3}\.){3}[0-9]{1,3}\b')

    for line in file_obj:
        # Handle bytes vs string when reading uploaded files in Colab
        if isinstance(line, bytes):
            line_str = line.decode("utf-8", errors="ignore").strip()
        else:
            line_str = line.strip()

        if not line_str:
            continue

        match_ip = ip_pattern.search(line_str)
        ip = match_ip.group(0) if match_ip else "Unknown"

        is_suspicious = any(kw in line_str.lower() for kw in ["failed", "unauthorized", "invalid", "sql syntax", "404", "403"])

        logs.append({
            "log_line": line_str,
            "ip_address": ip,
            "suspicious": is_suspicious
        })

    df = pd.DataFrame(logs)
    return df

Overwriting detector.py


In [ ]:
%%writefile log_generator.py
import time
import random
from datetime import datetime

ips = ["192.168.1.50", "10.0.0.12", "185.220.101.5", "203.0.113.19", "8.8.8.8", "172.16.254.1"]
activities = [
    "INFO - User admin logged in successfully",
    "WARNING - Failed login attempt for user root",
    "INFO - GET /index.html 200",
    "ERROR - Invalid credentials from user guest",
    "WARNING - SQL Syntax error near 'UNION SELECT'",
    "INFO - GET /images/logo.png 200",
    "ERROR - 404 Not Found - GET /admin-panel/login.php",
    "WARNING - Unauthorized access attempt to /etc/passwd"
]

print("Starting mock live log generator... writing to live_security.log")
with open("live_security.log", "w") as f:
    f.write(f"{datetime.now().isoformat()} - INFO - System Initialized\n")

while True:
    timestamp = datetime.now().isoformat()
    ip = random.choice(ips)
    activity = random.choice(activities)
    log_line = f"{timestamp} - IP: {ip} - {activity}\n"

    with open("live_security.log", "a") as f:
        f.write(log_line)

    time.sleep(random.uniform(1.0, 3.0))

Writing log_generator.py


In [ ]:
%%writefile app.py
import streamlit as st
from detector import parse_logs
import os
import time

st.set_page_config(page_title="Real-Time SIEM Threat Detector", layout="wide")

st.title("🛡️ Real-Time Log Analysis & Threat Detection Engine")
st.markdown("Tracking live suspicious IP activity on **live_security.log** instantly.")

log_file_path = "live_security.log"

# Fallback if the background generator hasn't started yet
if not os.path.exists(log_file_path):
    with open(log_file_path, "w") as f:
        f.write("System Initialized - Awaiting active stream...\n")

# Read logs from live file
with open(log_file_path, "r") as f:
    lines = f.readlines()

# Run detector
df = parse_logs(lines)
total_logs = len(df)
suspicious_count = int(df["suspicious"].sum())

col1, col2 = st.columns(2)
col1.metric("Total Logs Processed (Live)", total_logs)
col2.metric("Suspicious Events Flagged", suspicious_count)

# Show real-time IP alerts
st.subheader("🚨 Top Flagged Threat Sources (Real-time)")
if suspicious_count > 0:
    sus_df = df[df["suspicious"] == True]
    ip_counts = sus_df["ip_address"].value_counts().reset_index()
    ip_counts.columns = ["IP Address", "Flagged Events"]
    st.dataframe(ip_counts, use_container_width=True)
else:
    st.success("No suspicious activity detected in this stream yet!")

# Scroll-like Raw Log Inspector
st.subheader("Live Log stream (showing last 25 entries)")
st.dataframe(df.tail(25), use_container_width=True)

# Branding Footer
st.markdown("---")
st.markdown("<div style='text-align: center; color: gray;'>Engineered by <b>Wiltecon Technologies</b>, founded by <b>Wilson Nometa</b></div>", unsafe_allow_html=True)

# Trigger real-time visual refresh every 2 seconds
time.sleep(2)
st.rerun()

Overwriting app.py


In [ ]:
# Start the live log generator process in the background
import subprocess
subprocess.Popen(["python", "log_generator.py"])

<Popen: returncode: None args: ['python', 'log_generator.py']>

In [ ]:
%%writefile app.py
import streamlit as st
from detector import parse_logs
import os
import time

st.set_page_config(page_title="Real-Time SIEM Threat Detector", layout="wide")

st.title("🛡️ Real-Time Log Analysis & Threat Detection Engine")
st.markdown("Tracking live suspicious IP activity on **live_security.log** instantly.")

log_file_path = "live_security.log"

# Fallback if the background generator hasn't started yet
if not os.path.exists(log_file_path):
    with open(log_file_path, "w") as f:
        f.write("System Initialized - Awaiting active stream...\n")

# Read logs from live file
with open(log_file_path, "r") as f:
    lines = f.readlines()

# Run detector
df = parse_logs(lines)
total_logs = len(df)
suspicious_count = int(df["suspicious"].sum())

col1, col2 = st.columns(2)
col1.metric("Total Logs Processed (Live)", total_logs)
col2.metric("Suspicious Events Flagged", suspicious_count)

# Show real-time IP alerts
st.subheader("🚨 Top Flagged Threat Sources (Real-time)")
if suspicious_count > 0:
    sus_df = df[df["suspicious"] == True]
    ip_counts = sus_df["ip_address"].value_counts().reset_index()
    ip_counts.columns = ["IP Address", "Flagged Events"]
    st.dataframe(ip_counts, use_container_width=True)
else:
    st.success("No suspicious activity detected in this stream yet!")

# Scroll-like Raw Log Inspector
st.subheader("Live Log stream (showing last 25 entries)")
st.dataframe(df.tail(25), use_container_width=True)

# Branding Footer
st.markdown("---")
st.markdown("<div style='text-align: center; color: gray;'>Engineered by <b>Wiltecon Technologies</b>, founded by <b>Wilson Nometa</b></div>", unsafe_allow_html=True)

# Trigger real-time visual refresh every 2 seconds
time.sleep(2)
st.rerun()

Overwriting app.py


In [ ]:
# Run Streamlit in the background and use localtunnel to expose it
!streamlit run app.py & npx localtunnel --port 8501

⠙⠹⠸

⠼⠴⠦⠧Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) 2026-10-20 02:06:43.681 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.145.176.231:8501

  Stopping...
^C


In [ ]:
# Run Streamlit in background using Python module path and expose it using localtunnel
!python -m streamlit run app.py & npx -y localtunnel --port 8501

/bin/bash: line 1: streamlit: command not found
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼your url is: https://vast-ducks-ask.loca.lt
